# Lab 6: Credit scores and lending decisions

A historical benchmark, honest validation and explicit costs

> **Note**
>
> **Time:** 100 minutes for the core lab. The two extensions can follow
> as independent practice.
>
> Run the cells in order. Each calculation uses objects created in
> earlier cells. Read the explanation before running a cell, then use
> the output to answer the question at the end of the section.
>
> Attempt each question before revealing its sample answer. Sample
> answers appear on this HTML page and are omitted from the student
> notebook.
>
> <figure>
> <a
> href="https://colab.research.google.com/github/quinfer/fin510-colab-notebooks/blob/main/labs/lab06_alt_finance.ipynb"><img
> src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
> <figcaption>Open in Colab</figcaption>
> </figure>

A lending decision needs more than a risk ranking: it needs a
probability for the relevant population, loan terms and an account of
the errors that matter. We will compare a numerical-only classifier with
one that also uses categorical information, choose decision rules on
validation data, then evaluate both predictions and decisions on
untouched test records. The
[chapter](https://financialdatascience.co.uk/chapters/05_alt_finance_marketplace_lending.html)
explains the financial reasoning. A separate one-year cash-flow example
later in the lab connects default probabilities to investor returns.

## 1. Establish what one row and one label mean

The UCI Statlog German Credit file contains historical credit records.
We code its bad-credit label as 1 and good-credit label as 0. UCI’s
[corrected South German
documentation](https://archive.ics.uci.edu/dataset/522/south+german+credit)
dates the underlying records to 1973–1975 and describes oversampling of
bad credits and errors in the original coding (Hofmann 1994; Grömping
2019). The label does not establish a common default horizon. We can
practise classification here; current lending decisions require
representative applicants and a defined repayment outcome.

### Load the same data on each machine

We use pandas for tables, NumPy for arrays and Matplotlib for charts.
The other imports let the same loader read a local file or download the
pinned teaching copy.

In [ ]:
from pathlib import Path
from io import BytesIO
from urllib.request import urlopen
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

The loader prefers the course CSV if it is available locally. Otherwise
it downloads a fixed repository version, so the exercise is reproducible
in Colab. `raw_bytes` is the file’s content before pandas interprets its
columns.

In [ ]:
paths = [Path("data/alt_finance/german_credit.csv"),
         Path("../data/alt_finance/german_credit.csv"),
         Path("../../data/alt_finance/german_credit.csv"), Path("german_credit.csv")]
local = next((path for path in paths if path.is_file()), None)
source = str(local) if local else (
    "https://raw.githubusercontent.com/quinfer/fin510-colab-notebooks/"
    "027c830b4041ca07e1b298377a32ab0666b272d2/labs/german_credit.csv")
raw_bytes = local.read_bytes() if local else urlopen(source, timeout=30).read()

A checksum identifies this exact CSV. Stop if the check fails: changing
the file changes the experiment. The CSV calls its outcome `defaulted`;
we rename that column because the evidence supports a historical
bad-credit label, not a timed default event.

In [ ]:
expected_sha256 = "37ef283edd556d84793d24cc455b868e2deb2dfc7c437fc70f7825ec4b07a708"
assert hashlib.sha256(raw_bytes).hexdigest() == expected_sha256, "Unexpected dataset version"
df = pd.read_csv(BytesIO(raw_bytes)).rename(columns={"defaulted": "bad_credit"})
assert df.shape == (1000, 21)
assert df.bad_credit.value_counts().to_dict() == {0: 700, 1: 300}
assert not df.isna().any().any()
print("Source:", source, "| scikit-learn:", sklearn.__version__)
print("Rows:", len(df), "| input columns:", df.shape[1] - 1)
print(df.bad_credit.value_counts().rename_axis("Label").rename("Records"))

The output distinguishes the number of observations from the number of
input columns. The label counts describe the selected benchmark sample.
Their balance does not estimate a current lender’s default rate.

### Separate inputs from the outcome

`X` contains information supplied to the model; `y` contains the outcome
it must predict. Keeping the outcome out of `X` prevents the model from
being given the answer. We use column types to select the numerical and
categorical branches.

In [ ]:
X = df.drop(columns="bad_credit")
y = df.bad_credit
numeric = X.select_dtypes(include="number").columns.tolist()
categorical = X.select_dtypes(exclude="number").columns.tolist()
assert len(numeric) == 7 and len(categorical) == 13
print("Numerical inputs:", numeric)
print("Categorical inputs:", categorical)

Inspect those names before modelling. Some numerical fields are ordered
codes, so treating adjacent codes as equally spaced is an assumption.
The personal-status field mixes sex and marital-status information, and
the corrected documentation warns against reconstructing sex from it.
The amount field also has a documented transformation; it is not an
audited pound exposure.

**Q1.** Write a data specification covering row, label, horizon,
population and information available at prediction time. Which parts
cannot be recovered here? Why is this not P2P lending data?

## 2. Build a procedure that can assess a new applicant

A future applicant must be processed using rules learned from training
data. Scaling learns means and standard deviations; encoding learns
which categories exist. These operations belong inside the model’s
fitted procedure.

### Allocate the rows before learning from them

We first reserve one fifth of the rows for the final test. We then
reserve one quarter of the remaining rows for validation. `stratify`
keeps the label mix similar across the splits. `random_state` makes this
allocation reproducible; it does not establish that the historical
sample represents future borrowers.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

development, test = train_test_split(
    X.index, test_size=0.20, random_state=42, stratify=y)
train, validation = train_test_split(
    development, test_size=0.25, random_state=43, stratify=y.loc[development])
assert not (set(train) & set(validation) or set(train) & set(test)
            or set(validation) & set(test))
print(pd.Series({"Training": len(train), "Validation": len(validation),
                 "Final test": len(test)}, name="Rows"))

Read the three counts as three different jobs. Training estimates
preprocessing and coefficients. Validation chooses the rejection
threshold. The final test is used only after those choices have been
fixed. `.loc[train]` selects the records whose row identifiers are in
`train`.

### Describe the model before fitting it

Both models use the same seven numerical inputs. The all-input model
adds thirteen categorical fields. Standardisation puts numerical inputs
on a common training-data scale; one-hot encoding represents categories
by indicators. These coded historical categories do not constitute a
test of modern digital-footprint data.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

def make_model(include_categories):
    steps = [("numeric", StandardScaler(), numeric)]
    if include_categories:
        encoder = OneHotEncoder(drop="first", handle_unknown="ignore",
                                sparse_output=False)
        steps.append(("category", encoder, categorical))
    prepare = ColumnTransformer(steps, remainder="drop")
    logit = LogisticRegression(C=1, max_iter=3000)
    return Pipeline([("prepare", prepare), ("logit", logit)])

models = {"Numerical": make_model(False), "All inputs": make_model(True)}

`ColumnTransformer` sends each column to its stated preparation step.
`Pipeline` then passes the prepared inputs to logistic regression.
Creating these objects has not fitted them yet. Both use `C=1`, where a
smaller `C` means stronger coefficient penalisation.

`drop="first"` omits a reference category. `handle_unknown="ignore"`
maps an unseen category to zero indicators; that looks like the omitted
category. This fallback allows a prediction, but it does not show that
the unseen category has the same risk. Monitor how often it occurs.

### Evaluate the procedure within training folds

Each cross-validation round fits the entire pipeline on its training
portion and assesses the held-out portion. The scaler and encoder are
therefore relearned without looking at that fold’s held-out rows. AUC
assesses risk ranking; a value of 0.50 describes random ranking and 1.00
perfect ranking.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X.loc[train], y.loc[train],
                             cv=cv, scoring="roc_auc")
    cv_results[name] = scores
    print(name, "fold AUCs:", scores.round(3),
          "mean/SD:", np.round([scores.mean(), scores.std()], 3))

The variation across folds describes sensitivity to this training
allocation. It is not a confidence interval for a future lending
population. We now fit each pipeline once using all the training rows.
Validation and test rows remain outside this fit.

In [ ]:
for model in models.values():
    model.fit(X.loc[train], y.loc[train])
    assert model.classes_.tolist() == [0, 1]
print("Both models fitted on training rows only.")

**Q2.** Which operations learn from data? Explain why each must be
fitted within the training folds. Why keep `C=1` fixed in both models?
Suggest an alternative encoding for ordinal predictors.

## 3. Choose the decision rule before opening the test

We use a positive flag to mean predicted bad credit, followed by
rejection in this simplified policy. A false positive rejects a
good-credit record; a false negative approves a bad-credit record.
Lowering the rejection threshold catches more bad labels but rejects
more good labels too.

### Predict validation probabilities

`predict_proba` returns one column per class. We checked the class order
above, so column 1 gives the probability of the bad-credit label. The
slice `[:, 1]` takes that column for every validation record.

In [ ]:
from sklearn.metrics import (roc_auc_score, log_loss, brier_score_loss,
                             confusion_matrix, precision_recall_curve)
from sklearn.calibration import calibration_curve

validation_probability = models["All inputs"].predict_proba(X.loc[validation])[:, 1]
print("First five validation probabilities:", validation_probability[:5].round(3))

Probabilities are decimal fractions: 0.20 means 20%. These predictions
concern the historical label. They are not established one-year default
probabilities for a current lender.

### Compare candidate thresholds on validation data

Precision is the share of bad-credit flags that are correct. Recall is
the share of all bad-credit labels caught by the flags. F1 combines
precision and recall through their harmonic mean and ignores correctly
unflagged good-credit records. It does not include loan size, interest,
fees or recovery.

The curve function returns a list of possible thresholds and their
precision and recall. Its final precision/recall point has no
corresponding threshold, so `[:-1]` removes that point before we
calculate F1. The small denominator floor handles a candidate with both
precision and recall zero.

In [ ]:
pv, rv, thresholds = precision_recall_curve(y.loc[validation], validation_probability)
f1 = 2 * pv[:-1] * rv[:-1] / np.maximum(pv[:-1] + rv[:-1], 1e-12)
best_position = np.argmax(f1)
chosen_threshold = float(thresholds[best_position])
print("Selected validation threshold:", round(chosen_threshold, 3))
print("Validation precision, recall and F1:",
      np.round([pv[best_position], rv[best_position], f1[best_position]], 3))

`argmax` identifies the largest validation F1; it does not inspect test
outcomes. We compare this threshold with 0.50 and with a stated
error-cost rule. If approval of a bad label costs five units and
rejection of a good label costs one, expected approval cost is $5p$ and
expected rejection cost is $1-p$. Reject when $5p>1-p$, giving $p>1/6$;
the two actions have equal expected cost at the boundary.

In [ ]:
policies = {"Default 0.50": 0.50, "Validation F1": chosen_threshold,
            "Illustrative cost 1/6": 1 / 6}
print(pd.Series(policies, name="Fixed rejection threshold").round(3))

These are illustrative label-error costs in the sampled benchmark, not
observed financial losses or deployment advice. The three rules are now
fixed. The next section may assess them; it must not choose replacements
after seeing their test results.

**Q3.** Explain the access/capital trade-off using false positives and
false negatives. What would be wrong with maximising F1 on the test
labels and reporting that maximum as deployment performance?

## 4. Evaluate predictions on the same test records

We now open the final test. Comparing both models on the same rows keeps
differences in test composition out of the comparison. The model and
threshold choices are already fixed.

### Turn fitted models into test predictions

We retain one probability per model for each test row. `y_test` stores
those rows’ actual labels in the same order. For subsequent policy work,
`p` refers to the all-input model’s probabilities.

In [ ]:
y_test = y.loc[test].to_numpy()
probabilities = {}
for name, model in models.items():
    probabilities[name] = model.predict_proba(X.loc[test])[:, 1]
p = probabilities["All inputs"]
assert all(len(values) == len(y_test) for values in probabilities.values())

### Read ranking and probability accuracy separately

AUC measures how often a bad-credit record receives a higher risk score
than a good-credit record, counting ties as one half. Brier score
averages squared differences between probabilities and binary labels.
Log-loss strongly penalises assigning very little probability to an
outcome that occurs. Higher AUC and lower probability losses are better;
none of these measures is an approval rate.

In [ ]:
metric_rows = []
for name, values in probabilities.items():
    metric_rows.append({"Model": name, "AUC": roc_auc_score(y_test, values),
                        "log_loss": log_loss(y_test, values),
                        "Brier": brier_score_loss(y_test, values)})
metrics = pd.DataFrame(metric_rows).set_index("Model")
print(metrics.round(4).to_string())

Read across each row before choosing which result to emphasise. A model
can rank well yet give poorly calibrated probability levels. AUC of 0.8
is not 80% classification accuracy and does not mean 80% of applicants
will repay.

### Ask how uncertain the difference is

The bootstrap repeatedly samples test-row positions with replacement.
`idx` contains NumPy positions, which select the same entries from the
label array and both prediction arrays. Each draw carries those three
values together, preserving the paired comparison. Repeated rows are
expected; the models themselves are not refitted.

In [ ]:
boot_rng = np.random.default_rng(2026)
paired_difference = []
for _ in range(2000):
    idx = boot_rng.integers(0, len(test), len(test))
    if np.unique(y_test[idx]).size == 2:
        full_auc = roc_auc_score(y_test[idx], p[idx])
        numerical_auc = roc_auc_score(y_test[idx], probabilities["Numerical"][idx])
        paired_difference.append(full_auc - numerical_auc)
paired_interval = np.quantile(paired_difference, [0.025, 0.975])
print("Test AUC difference:", metrics.loc["All inputs", "AUC"] - metrics.loc["Numerical", "AUC"])
print("Paired-bootstrap 95% interval:", paired_interval)

The interval describes resampling uncertainty in this test comparison,
conditional on the fitted models. It does not include new model fits or
a different economic population. If you revise the model after seeing
these results, assess the revised procedure using a new untouched
evaluation sample.

### Compare probability levels with observed frequencies

We divide predictions into five similarly sized groups. For each group,
the calibration function returns mean predicted probability and observed
bad-label frequency. The counts show how much evidence supports each
point.

In [ ]:
observed, predicted = calibration_curve(y_test, p, n_bins=5, strategy="quantile")
counts = pd.Series(pd.qcut(p, 5, duplicates="drop")).value_counts(sort=False).to_numpy()
calibration_table = pd.DataFrame({"Mean predicted risk": predicted,
                                  "Observed bad-label share": observed,
                                  "Records": counts})
assert counts.sum() == len(test)
print(calibration_table.round(3).to_string(index=False))

A point below the diagonal has a lower observed bad-label share than the
predicted risk: the model overpredicts risk in that group. Sampling
noise makes small differences uncertain. We plot the same quantities
from the table, without recalibrating the model on these test outcomes.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([0, 1], [0, 1], "--", color="grey", label="Predicted = observed")
ax.plot(predicted, observed, "o-", color="steelblue")
for mean_risk, bad_share, count in zip(predicted, observed, counts):
    ax.annotate(f"n={count}", (mean_risk, bad_share), xytext=(4, 5),
                textcoords="offset points")
ax.set(xlabel="Predicted benchmark bad-label probability",
       ylabel="Observed bad-label share", xlim=(0, 1), ylim=(0, 1))
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

**Q4.** Use the printed results to recommend a model within this
benchmark. What does the paired interval include, and what uncertainty
does it omit? Interpret a calibration point below the diagonal.

## 5. Translate predictions into decisions and costs

A score supplies a probability; a threshold supplies an action. We first
inspect the confusion matrix for the cost-rule threshold. Rows are
actual labels and columns are the actions implied by the flags. A
positive flag leads to rejection.

In [ ]:
flagged = p >= policies["Illustrative cost 1/6"]
matrix = confusion_matrix(y_test, flagged)
decision_counts = pd.DataFrame(matrix, index=["Actual good", "Actual bad"],
                               columns=["Approve", "Reject"])
print(decision_counts)

The good/reject cell counts false positives. The bad/approve cell counts
false negatives. Adding the two approval cells gives the number
approved, while adding all four cells must recover the test size.

We now repeat that calculation for each rule fixed on validation.
`ravel()` reads the confusion matrix as true negatives, false positives,
false negatives and true positives. `~flagged` reverses the True/False
flags, and summing it counts approvals. Each policy changes the flags
while leaving the fitted probabilities unchanged.

In [ ]:
policy_rows = []
for name, threshold in policies.items():
    flagged = p >= threshold
    tn, fp, fn, tp = confusion_matrix(y_test, flagged).ravel()
    policy_rows.append({"policy": name, "threshold": threshold,
                        "approved": int((~flagged).sum()), "good_rejected": int(fp),
                        "bad_approved": int(fn), "precision": tp / max(tp + fp, 1),
                        "recall": tp / (tp + fn), "label_error_cost": int(fp + 5 * fn)})
policy_table = pd.DataFrame(policy_rows)
print(policy_table.round(3).to_string(index=False))

Read both error columns before interpreting cost. The five-to-one rule
minimises expected error cost under its assumptions, not necessarily the
realised cost in this finite test. Probabilities are approximate and are
calibrated, if at all, to the sampled label mix. Keep these units
separate from investor profit and borrower benefit.

**Q5.** Which policy rejects most good risks? Which approves most bad
risks? Explain why the cheapest benchmark policy might still be
unsuitable for a lender. Write a three-sentence model recommendation
with one uncertainty and one deployment limitation.

## 6. Calculate a return from stated cash flows

We now leave the historical labels. The following contracts are invented
one-year loans of £1. Their assumed default probabilities and coupon
interest rates are inputs, not estimates from German Credit. Fee f is 1%
of the initial principal and is paid on every loan at year end.

In [ ]:
contracts = pd.DataFrame({"scenario": ["Lower risk", "Intermediate", "Higher risk"],
                          "PD": [0.05, 0.15, 0.35], "coupon": [0.06, 0.15, 0.25]})
fee = 0.01
print(contracts.to_string(index=False))

Coupon r is earned only if the borrower repays. When default occurs, the
investor recovers fraction q of principal and receives no coupon. For
each £1 lent, expected interest is $(1-p)r$, expected principal loss is
$p(1-q)$ and expected net return is their difference minus the fee.

We calculate each component separately so you can trace the financial
logic. Increasing recovery changes the principal-loss term; it leaves
the coupon, fee and default probability fixed.

In [ ]:
for recovery in [0, 0.40]:
    expected_interest = (1 - contracts.PD) * contracts.coupon
    expected_principal_loss = contracts.PD * (1 - recovery)
    contracts[f"net_recovery_{recovery:.1f}"] = (
        expected_interest - expected_principal_loss - fee)
print(contracts.round(4).to_string(index=False))

The return columns are decimal returns on the initial £1. The chart
multiplies them by 100 to display percentages; it does not change the
horizon or annualise a multi-year loan.

In [ ]:
returns_to_plot = contracts.set_index("scenario").filter(like="net_").mul(100)
returns_to_plot = returns_to_plot.rename(columns={"net_recovery_0.0": "No recovery",
                                                 "net_recovery_0.4": "40% recovery"})
returns_to_plot.plot.bar(figsize=(8, 4), rot=0)
plt.axhline(0, color="grey")
plt.ylabel("Expected one-year net return (%)")
plt.tight_layout()
plt.show()

A lower default probability can still expose more pounds to loss if the
loan is larger. The following separate examples assume no recovery.
Compare the last column, not just the probabilities.

In [ ]:
principal_examples = pd.DataFrame({"PD": [0.10, 0.05], "Exposure (£)": [100, 1000]})
principal_examples["Expected principal loss (£)"] = (
    principal_examples.PD * principal_examples["Exposure (£)"])
print(principal_examples.to_string(index=False))

**Q6.** At a 15% coupon and 1% fee, calculate break-even PD with zero
and 40% recovery. Why does higher AUC not automatically imply higher
investor returns? What changes for a multi-year amortising loan?

## 7. Extension: adjust the assumed population prevalence

Prevalence is the share of records with an adverse label. The benchmark
deliberately oversamples bad-credit records. We do not know the
historical population prevalence; 5% below is a sensitivity assumption.

Suppose only the label mix changes: the characteristics of good-credit
records and of bad-credit records each retain their distributions. Under
that assumption, we adjust the odds using the ratio of target-population
base-rate odds to sample base-rate odds. Converting the adjusted odds
back to probability gives the formula in this function.

In [ ]:
def prior_adjust(probability, sample_prevalence, target_prevalence):
    p = np.asarray(probability, dtype=float)
    if not (0 < sample_prevalence < 1 and 0 < target_prevalence < 1):
        raise ValueError("Prevalences must lie strictly between zero and one")
    if not np.all(np.isfinite(p) & (p >= 0) & (p <= 1)):
        raise ValueError("Probabilities must be finite and between zero and one")
    multiplier = (target_prevalence / (1 - target_prevalence)) / (
        sample_prevalence / (1 - sample_prevalence))
    return multiplier * p / (1 - p + multiplier * p)

The checks admit predicted probabilities at 0 and 1 but require the two
base rates to lie strictly between them. We first apply the
transformation to three illustrative predictions. Inspect whether the
adjustment subtracts a fixed percentage or instead changes the
probability scale.

In [ ]:
sample_predictions = np.array([0.10, 0.30, 0.60])
adjustment_example = pd.DataFrame({"Sample prediction": sample_predictions,
    "Adjusted prediction": prior_adjust(sample_predictions, 0.30, 0.05)})
print(adjustment_example.round(4).to_string(index=False))

We then transform the actual all-input test predictions. AUC on these
same rows should remain unchanged because the transformation preserves
their ordering. Flags at a fixed numerical threshold can change. This is
not a validation of calibration in an unobserved target population.

In [ ]:
adjusted = prior_adjust(p, 0.30, 0.05)
print("Same-row AUC before/after:", roc_auc_score(y_test, p),
      roc_auc_score(y_test, adjusted))
print("Rejection flags at 1/6 before/after:",
      (p >= 1 / 6).sum(), (adjusted >= 1 / 6).sum())

**Q7.** Why does this transformation preserve ranking on these rows but
change decisions? Which assumptions would fail if loan approval selected
on income as well as risk? Does the transformed probability’s
calibration on this original sample establish target-population
calibration?

## 8. Extension: compare groups using stated denominators

We use the recorded age field for a descriptive audit at the same fixed
threshold. This does not identify discrimination or establish the legal
permissibility of a predictor. Other personal-status codes in the file
are not a reliable basis for reconstructing sex groups.

### Assemble one aligned audit row per test record

We take age from the same test rows used for the labels and predictions.
The bins include ages from 30 up to, but not including, 50 in the middle
group: `right=False` makes the boundaries agree with the labels.

In [ ]:
audit = X.loc[test, ["age"]].copy()
audit["bad_credit"] = y_test
audit["probability"] = p
audit["rejected"] = p >= 1 / 6
audit["group"] = pd.cut(audit.age, [0, 30, 50, np.inf], right=False,
                         labels=["Under 30", "30 to 49", "50 and over"])
assert len(audit) == len(test) and audit.group.notna().all()
print(audit.groupby("group", observed=True).size().rename("Test records"))

### Count the population relevant to each rate

The rejection share divides by all records in a group. False-positive
rate divides by good-credit records only; false-negative rate divides by
bad-credit records only. We retain both denominators in the table so
that a large percentage based on a few records is recognisable.

In [ ]:
group_rows = []
for name, group in audit.groupby("group", observed=True):
    bad = group.bad_credit == 1
    good = ~bad
    group_rows.append({"group": name, "n": len(group), "bad_n": int(bad.sum()),
                       "good_n": int(good.sum()), "bad_share": bad.mean(),
                       "mean_probability": group.probability.mean(),
                       "rejection_share": group.rejected.mean(),
                       "false_positive_rate": group.loc[good, "rejected"].mean(),
                       "false_negative_rate": (~group.loc[bad, "rejected"]).mean()})
group_summary = pd.DataFrame(group_rows)
assert group_summary.n.sum() == len(test)
print(group_summary.round(3).to_string(index=False))

Read `n`, `good_n` and `bad_n` before comparing percentages. Comparing
mean predicted risk with observed bad-label share is a coarse
calibration check. A difference in rejection shares describes access
under this policy. Neither comparison alone explains why a group
difference arises. Monitoring new applications can reveal changed inputs
before enough time has passed to assess repayment outcomes.

**Q8.** Distinguish a group calibration gap, a rejection-rate gap and an
error-rate gap. Which comparisons have small denominators? What would
you monitor prospectively if applicant mix changed before repayment
labels became available?

Grömping, Ulrike. 2019. “South German Credit.” UCI Machine Learning
Repository.
<https://archive.ics.uci.edu/dataset/522/south+german+credit>.

Hofmann, Hans. 1994. “Statlog (German Credit Data).” UCI Machine
Learning Repository. <https://doi.org/10.24432/C5NC77>.